# Heart Disease Risk Prediction & Clinical Model Comparison

## 1. Project Overview & Clinical Problem Statement
Cardiovascular diseases (CVDs) remain the leading cause of global mortality. Early, accurate risk identification enables proactive clinical interventions. In screening and triage settings, **minimizing False Negatives (missed cases of disease)** is critical, as failing to identify a patient with heart disease carries significantly higher morbidity and mortality risks than conducting secondary confirmatory testing on a false positive.

> **Disclaimer:** This notebook is an educational/research demonstration. It is not a medical diagnostic tool and must not be used for healthcare decision-making.

In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Ensure project root is on Python path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_processed_data, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, ALL_FEATURES
from src.preprocessing import create_full_pipeline
from src.evaluate import evaluate_model, build_comparison_dataframe, analyze_thresholds

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries successfully imported.")

## 2. Dataset Ingestion & Exploration
We load the UCI Heart Disease dataset (Cleveland database, ID 45). The target is converted into binary format where 0 indicates absence of disease and 1 indicates presence.

In [ ]:
X, y = load_processed_data()
df = pd.concat([X, y], axis=1)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
print("Missing Value Count:")
print(df.isnull().sum())
print("\nSummary Statistics:")
df.describe().T

## 3. Exploratory Data Analysis & Clinical Correlations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Target Distribution
sns.countplot(x='target', data=df, ax=axes[0], palette=['#2b5c8f', '#d9534f'])
axes[0].set_title('Target Class Distribution (0=No Disease, 1=Disease)', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['No Disease (0)', 'Disease (1)'])

# Chest Pain Type vs Disease
sns.countplot(x='cp', hue='target', data=df, ax=axes[1], palette=['#2b5c8f', '#d9534f'])
axes[1].set_title('Heart Disease by Chest Pain Type (1: Typ, 2: Atyp, 3: Non-anginal, 4: Asymptomatic)', fontsize=12, fontweight='bold')
axes[1].legend(['No Disease', 'Disease'])
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap for Numerical Attributes
plt.figure(figsize=(9, 7))
corr = df[NUMERICAL_FEATURES + ['target']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Key Clinical Attributes', fontsize=13, fontweight='bold')
plt.show()

## 4. Stratified Train / Test Split & Pipeline Modeling
We use stratified splitting to preserve class proportions and encapsulate preprocessing inside scikit-learn Pipelines to guarantee zero leakage.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced', max_depth=5),
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss')
}

results = []
pipelines = {}
for name, clf in models.items():
    pipe = create_full_pipeline(clf)
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe
    res = evaluate_model(pipe, X_test, y_test, model_name=name, threshold=0.5)
    results.append(res)

comp_df = build_comparison_dataframe(results)
comp_df

## 5. ROC Curves & False Negative Sensitivity Analysis

In [ ]:
plt.figure(figsize=(8, 6))
for res, color in zip(results, ['#2b5c8f', '#e67e22', '#27ae60']):
    plt.plot(res['fpr'], res['tpr'], lw=2.5, color=color, label=f"{res['model_name']} (AUC = {res['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('Recall / True Positive Rate')
plt.title('ROC Curves Comparison', fontsize=13, fontweight='bold')
plt.legend()
plt.show()

## 6. Threshold Optimization for Clinical Risk (Recall Maximization)

In [ ]:
rf_pipe = pipelines['Random Forest']
thresh_df = analyze_thresholds(rf_pipe, X_test, y_test, model_name='Random Forest')
thresh_df